In [9]:
import networkx as nx
import numpy as np
import scipy as sp
import pandas as pd
from itertools import combinations

ppi = pd.read_csv("4932.protein.links.v12.0.txt", sep=r"\s+")

# Filter for STRING scores >= 750
ppi_filtered = ppi[ppi["combined_score"] >= 750].copy()

# Construct network
G = nx.from_pandas_edgelist(
    ppi_filtered,
    source="protein1",
    target="protein2"
)

print("Proteins:", G.number_of_nodes())
print("Interactions:", G.number_of_edges())

Proteins: 5682
Interactions: 91503


In [10]:
#Determines the frequency an intermediate protein lies on the shortest paths between all three targets and high-centrality proteins
def intermediate_frequencies(G, source, target, min_distance=3):

    # Shortest-path distances from both endpoints
    d_source = nx.single_source_shortest_path_length(G, source)
    d_target = nx.single_source_shortest_path_length(G, target)

    if target not in d_source:
        raise ValueError(f"No path between {source} and {target}")

    # Count shortest paths from an origin to every reachable node
    def count_paths(origin, distances):
        counts = {origin: 1}

        for v in sorted(distances, key=distances.get):
            for w in G.neighbors(v):
                if distances.get(w) == distances[v] + 1:
                    counts[w] = counts.get(w, 0) + counts[v]

        return counts

    count_source = count_paths(source, d_source)
    count_target = count_paths(target, d_target)

    total_paths = count_source[target]
    shortest_distance = d_source[target]

    results = []

    for v in d_source:

        # Exclude endpoints
        if v in (source, target):
            continue

        if v not in d_target:
            continue

        # Check whether v lies on any shortest path
        if d_source[v] + d_target[v] != shortest_distance:
            continue

        # Apply minimum distance requirement
        if d_target[v] < min_distance:
            continue

        frequency = count_source[v] * count_target[v]

        results.append({
            "protein": v,
            "frequency": frequency,
            "fraction": frequency / total_paths,
            "distance_to_target": d_target[v]
        })

    df = pd.DataFrame(
        results,
        columns=[
            "protein", "frequency",
            "fraction", "distance_to_target"
        ]
    )

    return df.sort_values(
        "frequency", ascending=False
    ), total_paths

In [11]:
#Finding all intermediate proteins on the shortest paths between the high centrality protein and all three targets

# Selected high-centrality protein. Change when necessary!
source = "4932.YNL015W"  

#Target Proteins
HSC82 = "4932.YMR186W"
SSA1 = "4932.YAL005C"
HSF1 = "4932.YGL073W"

# HSC82
freq_HSC82, total_HSC82 = intermediate_frequencies(G, source, HSC82, min_distance=3)

# SSA1
freq_SSA1, total_SSA1 = intermediate_frequencies(G, source, SSA1, min_distance=3)

#HSF1
freq_HSF1, total_HSF1 = intermediate_frequencies(G, source, HSF1, min_distance=3)

print("Total shortest paths to HSC82:", total_HSC82)
print("Total shortest paths to SSA1:", total_SSA1)
print("Total shortest paths to HSF1:", total_HSF1)  

print("\nAll intermediates to HSC82:")
display(freq_HSC82)

print("\nAll intermediates to SSA1:")
display(freq_SSA1)

print("\nAll intermediates to HSF1:")
display(freq_HSF1)

Total shortest paths to HSC82: 94
Total shortest paths to SSA1: 92
Total shortest paths to HSF1: 9

All intermediates to HSC82:


,protein,frequency,fraction,distance_to_target
0,4932.YHR138C,94,1.000000,4
2,4932.YLR178C,93,0.989362,3
1,4932.YMR174C,1,0.010638,3



All intermediates to SSA1:


,protein,frequency,fraction,distance_to_target
0,4932.YHR138C,92,1.00000,4
2,4932.YLR178C,91,0.98913,3
1,4932.YMR174C,1,0.01087,3



All intermediates to HSF1:


,protein,frequency,fraction,distance_to_target
0,4932.YHR138C,9,1.0,4
1,4932.YLR178C,9,1.0,3


In [12]:
#Finding the frequency the intermediate protein lies on the shortest path from each target to high centrality protein

common_df = freq_HSC82.merge(
    freq_SSA1,
    on="protein",
    how="inner",
    suffixes=("_HSC82", "_SSA1")
)

# Exclude either target from intermediate candidates
common_df = common_df[
    ~common_df["protein"].isin([HSC82, SSA1, source])
].copy()

common_df["shared_score"] = common_df[
    ["fraction_HSC82", "fraction_SSA1"]
].min(axis=1)

# Rank by shared score
common_df = common_df.sort_values(
    "shared_score",
    ascending=False
)

display(common_df)



,protein,frequency_HSC82,fraction_HSC82,distance_to_target_HSC82,frequency_SSA1,fraction_SSA1,distance_to_target_SSA1,shared_score
0,4932.YHR138C,94,1.000000,4,92,1.00000,4,1.000000
1,4932.YLR178C,93,0.989362,3,91,0.98913,3,0.989130
2,4932.YMR174C,1,0.010638,3,1,0.01087,3,0.010638


In [14]:
#Target pairwise results of shared intermediate proteins

# Frequency tables for each target
freq_tables = {
    "HSC82": freq_HSC82,
    "SSA1": freq_SSA1,
    "HSF1": freq_HSF1
}

# Exclude targets and source from intermediate candidates
excluded = {HSC82, SSA1, HSF1, source}

# Store the results for each pair
pairwise_results = {}

for (name_a, df_a), (name_b, df_b) in combinations(freq_tables.items(), 2):

    # Find intermediate proteins shared by both targets
    common_df = df_a.merge(
        df_b,
        on="protein",
        how="inner",
        suffixes=(f"_{name_a}", f"_{name_b}")
    )

    # Exclude targets and source
    common_df = common_df[
        ~common_df["protein"].isin(excluded)
    ].copy()

    # Calculate shared score
    common_df["shared_score"] = common_df[
        [f"fraction_{name_a}", f"fraction_{name_b}"]
    ].min(axis=1)

    # Rank by shared score
    common_df = common_df.sort_values(
        "shared_score",
        ascending=False
    )

    # Save results
    pair_name = f"{name_a}_{name_b}"
    pairwise_results[pair_name] = common_df

    print(f"\n{name_a} vs {name_b}")
    display(common_df)


HSC82 vs SSA1


,protein,frequency_HSC82,fraction_HSC82,distance_to_target_HSC82,frequency_SSA1,fraction_SSA1,distance_to_target_SSA1,shared_score
0,4932.YHR138C,94,1.000000,4,92,1.00000,4,1.000000
1,4932.YLR178C,93,0.989362,3,91,0.98913,3,0.989130
2,4932.YMR174C,1,0.010638,3,1,0.01087,3,0.010638



HSC82 vs HSF1


,protein,frequency_HSC82,fraction_HSC82,distance_to_target_HSC82,frequency_HSF1,fraction_HSF1,distance_to_target_HSF1,shared_score
0,4932.YHR138C,94,1.000000,4,9,1.0,4,1.000000
1,4932.YLR178C,93,0.989362,3,9,1.0,3,0.989362



SSA1 vs HSF1


,protein,frequency_SSA1,fraction_SSA1,distance_to_target_SSA1,frequency_HSF1,fraction_HSF1,distance_to_target_HSF1,shared_score
0,4932.YHR138C,92,1.00000,4,9,1.0,4,1.00000
1,4932.YLR178C,91,0.98913,3,9,1.0,3,0.98913


In [17]:
#removes intermediate protein node
def node_removal_effect(G, source, target, intermediate):

    # Original shortest-path distance
    original_distance = nx.shortest_path_length(
        G, source, target
    )

    # Temporarily remove the intermediate protein
    G_removed = G.copy()
    G_removed.remove_node(intermediate)

    # Recalculate shortest-path distance
    try:
        new_distance = nx.shortest_path_length(
            G_removed, source, target
        )
        increase = new_distance - original_distance

    except nx.NetworkXNoPath:
        new_distance = float("inf")
        increase = float("inf")

    return original_distance, new_distance, increase

In [18]:
#Finding changes and the average los in shortest path from target to high centrality protein following removal of intermediate protein

targets = {
    "HSC82": HSC82,
    "SSA1": SSA1,
    "HSF1": HSF1
}

# Combine candidate proteins from all pairwise comparisons
candidates = set().union(
    *(set(df["protein"]) for df in pairwise_results.values())
)

# Exclude source and targets from candidates
candidates -= {source, *targets.values()}

# Original distances from source to all targets
original_distances = nx.single_source_shortest_path_length(
    G, source
)

results = []

for protein in candidates:

    # Remove intermediate protein without modifying G
    G_removed = nx.restricted_view(
        G, [protein], []
    )

    # Calculate all distances from source after removal
    new_distances = nx.single_source_shortest_path_length(
        G_removed, source
    )

    row = {"protein": protein}

    for name, target in targets.items():

        original = original_distances[target]
        new = new_distances.get(target, float("inf"))

        row[f"increase_{name}"] = new - original

        # Relative loss of shortest-path efficiency
        row[f"loss_{name}"] = 1 - original / new

    # Average connectivity loss across all three targets
    row["overall_loss"] = sum(
        row[f"loss_{name}"] for name in targets
    ) / len(targets)

    results.append(row)

removal_df = pd.DataFrame(results)

removal_df = removal_df.sort_values(
    "overall_loss",
    ascending=False
)

display(removal_df)

,protein,increase_HSC82,loss_HSC82,increase_SSA1,loss_SSA1,increase_HSF1,loss_HSF1,overall_loss
2,4932.YHR138C,inf,1.0,inf,1.0,inf,1.000000,1.000000
0,4932.YLR178C,0.0,0.0,0.0,0.0,1.0,0.166667,0.055556
1,4932.YMR174C,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000
